In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import numpy as np
import QuantLib as ql
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

In [ ]:
print(device)

In [ ]:
def generate_asset_paths_merton(S0, mu, sigma, T, M, N, lamb, mu_j, sigma_j, seed=42, device=device):
    """
    Generate asset paths using the Merton Jump-Diffusion model.

    Parameters:
    - S0: Initial stock price.
    - mu: Drift (annual return of the asset).
    - sigma: Volatility of the asset.
    - T: Time to maturity (in years).
    - M: Number of paths.
    - N: Number of time steps.
    - lamb: Jump intensity (average number of jumps per year).
    - mu_j: Mean of the jump size (log-normal distribution).
    - sigma_j: Volatility of the jump size.
    - device: Device to run the simulation on ('cpu' or 'cuda').

    Returns:
    - paths: A tensor of shape (M, N+1) containing the simulated asset paths.
    """
    torch.manual_seed(seed)
    dt = T / N
    # Precompute constants for efficiency
    drift = (mu - 0.5 * sigma ** 2) * dt
    diffusion = sigma * torch.sqrt(torch.tensor(dt, device=device))

    # Initialize the paths tensor
    paths = torch.zeros((M, N + 1), device=device)
    paths[:, 0] = S0

    # Generate random normal values for the Brownian motion
    Z = torch.randn((M, N), device=device)

    # Generate Poisson-distributed random numbers for the jumps
    J = torch.poisson(lamb * dt * torch.ones((M, N), device=device))

    # Generate the jump sizes (log-normal distributed)
    jump_sizes = torch.exp(mu_j + sigma_j * torch.randn((M, N), device=device)) - 1

    # Calculate the asset paths with jumps
    for t in range(1, N + 1):
        # Jumps impact: paths[:, t - 1] * (1 + jump_sizes[:, t - 1] * J[:, t - 1])
        paths[:, t] = paths[:, t - 1] * torch.exp(drift + diffusion * Z[:, t - 1]) * (1 + jump_sizes[:, t - 1] * J[:, t - 1])

    return paths

In [ ]:
def BS_CALL(S, K, T, r, sigma):

    # Calculate d1 and d2 using PyTorch
    d1 = (torch.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)
    

    # Calculate the call option price using the Black-Scholes formula
    CDF = torch.distributions.Normal(0, 1).cdf
    call_price = S * CDF(d1) - K * torch.exp(-r * T) * CDF(d2)
    return call_price

def merton_jump_call(S, K, T, r, sigma, m, v, lam):
    T = torch.clamp(T, min=1e-10)  # Clamp T to avoid division by zero or log issues
    p = torch.zeros_like(S)
    mu_j = np.exp(m+v**2*0.5)
    for k in range(40):
        r_k = r - lam * (mu_j - 1) + (k * torch.log(torch.tensor(mu_j, device=S.device))) / T
        sigma_k = torch.sqrt(sigma ** 2 + (k * v ** 2) / T)
        k_fact = torch.exp(torch.lgamma(torch.tensor(k + 1, device=S.device)))
        p += (torch.exp(-m * lam * T) * (mu_j * lam * T) ** k / k_fact) * BS_CALL(S, K, T, r_k, sigma_k)
    
    return p

def get_option_prices_and_deltas(stock_paths, K, r, sigma, T, N, m, v, lam, device='cpu'):
    stock_paths = stock_paths.to(device)
    M = stock_paths.size(0)  # Number of paths
    dt = T / N  # Time increment

    # Time to maturity for each time step
    time_to_maturity = torch.linspace(T, 0, steps=N+1, device=device).unsqueeze(0).expand(M, N+1)

    # Calculate option prices using the Merton Jump-Diffusion model
    option_prices = torch.zeros_like(stock_paths)
    for i in range(N+1):
        option_prices[:, i] = merton_jump_call(stock_paths[:, i], K, time_to_maturity[:, i], r, sigma, m, v, lam)

    # Calculate deltas using finite difference method
    epsilon = 1e-4  # Small increment for finite difference
    option_deltas = torch.zeros_like(stock_paths)
    for i in range(N+1):
        # Shift stock prices by a small epsilon
        shifted_stock_paths = stock_paths[:, i] + epsilon
        shifted_option_prices = merton_jump_call(shifted_stock_paths, K, time_to_maturity[:, i], r, sigma, m, v, lam)
        
        # Calculate delta as the difference in option prices divided by epsilon
        option_deltas[:, i] = (shifted_option_prices - option_prices[:, i]) / epsilon

    return option_prices, option_deltas

In [ ]:
def proportional_cost(transactions, proportion=0.01):
    return proportion * torch.abs(transactions)

In [ ]:
# Loss functions
def calculate_loss(pnl, loss_type='cvar', alpha=0.95, lmda = 1.0):
    if loss_type == 'cvar':
        sorted_pnl, _ = torch.sort(pnl)
        var_index = int((1 - alpha) * len(sorted_pnl))
        cvar = torch.mean(sorted_pnl[:var_index])
        return -cvar
    elif loss_type == 'entropic':
        return torch.log(torch.mean(torch.exp(-lmda * pnl))) / lmda
    elif loss_type == 'mean-variance':
        mean_pnl = torch.mean(pnl)
        variance_pnl = torch.var(pnl)
        return -mean_pnl + variance_pnl
    elif loss_type == 'mse':
        mse = torch.mean(pnl**2)
        return mse
    else:
        raise ValueError("Invalid loss_type.")

In [ ]:
class HedgingAgent(nn.Module):
    def __init__(self):
        super(HedgingAgent, self).__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=32, num_layers=2, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, state):
        # Assume state has shape (batch_size, sequence_length, input_size)
        lstm_out, _ = self.lstm(state)
        # We take the output from the last time step
        x = lstm_out[:, -1, :]
        return self.fc(x)

In [ ]:
def train_agent(agent, optimizer, asset_paths, option_prices, option_deltas, K, r, 
                sigma, T, N, transaction_cost_model, calculate_loss, batch_size=32, epochs=1, window_size=4, device=device):
    delta_T = T / N
    gamma = np.exp(-r * delta_T)  # Discount factor for reward
    agent.to(device)
    loss_lst = list()

    for iepoch in range(epochs):
        for pathblk in trange(len(asset_paths) // batch_size, desc="Training Epoch" + str(iepoch)):
            batch_start = pathblk * batch_size
            batch_end = batch_start + batch_size
            paths_batch = asset_paths[batch_start:batch_end]
            option_prices_batch = option_prices[batch_start:batch_end]
            option_deltas_batch = option_deltas[batch_start:batch_end]

            optimizer.zero_grad()

            # Initialize tensors for portfolio values and hedge positions
            portfolio_values = torch.zeros(batch_size, dtype=torch.float32).to(device)
            cash = torch.zeros(batch_size, dtype=torch.float32).to(device)
            stock_value = torch.zeros(batch_size, dtype=torch.float32).to(device)
            hedge_positions = torch.zeros(batch_size, dtype=torch.float32).to(device)

            # Iterate over each time step in parallel
            # buy the option
            cash -= option_prices_batch[:, 0]
            for t in range(N - window_size):
                toffset = t + window_size
                # Create states for the current window
                states = []
                for i in range(window_size):
                    state = torch.stack([paths_batch[:, t + i], hedge_positions, torch.full((batch_size,), T - (t + i) * delta_T, dtype=torch.float32).to(device)], dim=1)
                    states.append(state)
                states = torch.stack(states, dim=1)  # Shape: (batch_size, window_size, 3)
                hedge_actions = agent(states).squeeze()

                dHedge = hedge_actions - hedge_positions        
                transaction_costs = transaction_cost_model(dHedge)
                cash -= transaction_costs + dHedge * paths_batch[:, toffset]
                hedge_positions = hedge_actions
                stock_value = hedge_positions * paths_batch[:, toffset] 

                portfolio_values = cash + stock_value + option_prices_batch[:, toffset]

            # Calculate loss and perform backpropagation
            loss = calculate_loss(portfolio_values)
            loss.backward()
            loss_lst.append(loss.item())
            optimizer.step()
        
    return loss_lst


In [ ]:
def compare_strategies(agent, asset_paths, option_prices, option_deltas, K, r, 
                       sigma, T, N, transaction_cost_model, batch_size=32, window_size=4, device=device):
    agent.to(device)
    agent.eval()  # Set the agent to evaluation mode

    agent_pnl = []
    agent_sv = []
    bs_pnl = []
    bs_sv = []
    
    for epoch in trange(len(asset_paths) // batch_size, desc="Evaluating"):
        batch_start = epoch * batch_size
        batch_end = batch_start + batch_size
        paths_batch = asset_paths[batch_start:batch_end]
        option_prices_batch = option_prices[batch_start:batch_end]
        option_deltas_batch = option_deltas[batch_start:batch_end]

        # Initialize portfolio values and hedge positions for the entire batch
        portfolio_values = torch.zeros(batch_size, N, dtype=torch.float32).to(device)
        hedge_positions = torch.zeros(batch_size, dtype=torch.float32).to(device)
        cash = torch.zeros(batch_size, dtype=torch.float32).to(device)
        cash -= option_prices_batch[:, 0]
        stock_value = torch.zeros(batch_size, N, dtype=torch.float32).to(device)

        bs_portfolio_values = torch.zeros(batch_size, N, dtype=torch.float32).to(device)
        bs_hedge_positions = torch.zeros(batch_size, dtype=torch.float32).to(device)
        bs_cash = torch.zeros(batch_size, dtype=torch.float32).to(device)
        bs_cash-= option_prices_batch[:, 0]
        bs_stock_value = torch.zeros(batch_size, N, dtype=torch.float32).to(device)    

        for t in range(N - window_size):
            toffset = t + window_size
            # Create states for the current window
            states = []
            for i in range(window_size):
                time_to_maturity = T - (t + i) * (T / N)
                state = torch.stack([paths_batch[:, t + i], hedge_positions, torch.full((batch_size,), time_to_maturity, dtype=torch.float32).to(device)], dim=1)
                states.append(state)
            states = torch.stack(states, dim=1)  # Shape: (batch_size, window_size, 3)
            
            hedge_actions = agent(states).squeeze()

            dHedge = hedge_actions - hedge_positions        
            transaction_costs = transaction_cost_model(dHedge)
            cash -= transaction_costs + dHedge * paths_batch[:, toffset]
            hedge_positions = hedge_actions
            stock_value[:, toffset] = hedge_positions * paths_batch[:, toffset] 
            
            portfolio_values[:, toffset] = cash + stock_value[:, toffset] + option_prices_batch[:, toffset]
                    
        for t in range(N):
            # Black-Scholes hedge calculation
            bs_hedge_actions = -option_deltas_batch[:, t]
            bs_dHedge = bs_hedge_actions - bs_hedge_positions        
            bs_transaction_costs = transaction_cost_model(bs_dHedge)
            bs_cash -= bs_transaction_costs + bs_dHedge * paths_batch[:, t]
            bs_hedge_positions = bs_hedge_actions
            bs_stock_value[:, t] = bs_hedge_positions * paths_batch[:, t] 
            
            bs_portfolio_values[:, t] = bs_cash + bs_stock_value[:, t] + option_prices_batch[:, t]

        # Accumulate PnL for the batch
        agent_pnl.append(portfolio_values.detach().cpu().numpy())
        bs_pnl.append(bs_portfolio_values.detach().cpu().numpy())
        agent_sv.append(stock_value.detach().cpu().numpy())
        bs_sv.append(bs_stock_value.detach().cpu().numpy())
                
    return np.concatenate(agent_pnl, axis=0), np.concatenate(bs_pnl, axis=0), \
          np.concatenate(agent_sv, axis=0), np.concatenate(bs_sv, axis=0)


In [ ]:
# Hyperparameters
S0 = 1.0
mu = 0.00
sigma = 0.2
lamb = 1.0
mu_j = 0.0
sigma_j = 0.1
K = 1.0
r = 0.00
T = 1.0
M = 2**16
N = 50
batch_size = 256

In [ ]:
# Generate asset paths using Monte Carlo simulation
paths = generate_asset_paths_merton(S0, mu, sigma, T, M, N, lamb, mu_j, sigma_j)

In [ ]:
def plot_path(paths, index, T):
    """
    Plot a specific path from the array of simulated paths.

    Parameters:
    - paths: A tensor of shape (M, N+1) containing the simulated asset paths.
    - index: The index of the path to plot.
    - T: Time to maturity (in years).
    """
    # Extract the specific path
    path = paths[index].cpu().numpy()

    # Number of time steps
    N = path.shape[0] - 1

    # Time array
    time = torch.linspace(0, T, N+1).cpu().numpy()

    # Plot the path
    plt.figure(figsize=(10, 6))
    plt.plot(time, path, label=f'Path {index}')
    plt.title(f'Simulated Asset Path {index}')
    plt.xlabel('Time (Years)')
    plt.ylabel('Asset Price')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_path(paths, index=1, T=T)

In [ ]:
option_prices, option_deltas = get_option_prices_and_deltas(paths, K, r, sigma, T, N, 
                                                            mu_j, sigma_j, lamb, device=device)

In [ ]:
# Generate asset paths using Monte Carlo simulation
evalpaths = generate_asset_paths_merton(S0, mu, sigma, T, 2**14, N,  lamb, mu_j, sigma_j, seed=1001)

# Calculate option prices and deltas on the specified device
evaloption_prices, evaloption_deltas = get_option_prices_and_deltas(evalpaths, K, r, sigma, T, N, mu_j, sigma_j, lamb, device=device)

In [ ]:
def plot_pnl_histogram(agent_pnl, bs_pnl, filename, grayscale=False, bins=50):
    """
    Plots the P&L distributions of the trained agent and the Black-Scholes strategy as histograms on the same axes,
    ensuring they share the same bins, allowing for either color or grayscale output.

    Parameters:
    - agent_pnl: List or numpy array of P&L values for the trained agent.
    - bs_pnl: List or numpy array of P&L values for the Black-Scholes strategy.
    - filename: String, the filename where the plot will be saved.
    - bins: Number of bins for the histogram. Default is 50.
    - grayscale: Boolean, whether to plot in grayscale or color. Default is False (color).
    """
    plt.figure(figsize=(6, 4))
    
    # Determine color settings based on the grayscale flag
    if grayscale:
        agent_color, bs_color = 'black', 'gray'
    else:
        agent_color, bs_color = 'blue', 'orange'
    
    # Calculate the combined range of data for consistent binning
    min_val = min(np.min(agent_pnl), np.min(bs_pnl))
    max_val = max(np.max(agent_pnl), np.max(bs_pnl))
    
    # Create shared bins
    bins = np.linspace(min_val, max_val, bins)
    
    # Plot histogram for the agent's P&L with shared bins
    plt.hist(agent_pnl, bins=bins, alpha=0.5, label='Agent P&L', color=agent_color, 
             edgecolor='black', density=True)
    
    # Plot histogram for the Black-Scholes P&L with shared bins
    plt.hist(bs_pnl, bins=bins, alpha=0.5, label='MJD P&L', color=bs_color, 
             edgecolor='black', density=True)
    
    # Add titles and labels
    plt.title('P&L Distribution Comparison')
    plt.xlabel('P&L')
    plt.ylabel('Density')
    
    # Add a legend
    plt.legend(loc='upper right')
    
    # Save the plot
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    
    # Show the plot
    plt.grid(True)
    plt.show()

In [ ]:
# Initialize the agent and optimizer
agent = HedgingAgent().to(device)
optimizer = optim.Adam(agent.parameters(), lr=0.01)

In [ ]:
cl = lambda x: calculate_loss(x, 'mse')
train_loss = train_agent(agent, optimizer, paths, option_prices, option_deltas, K, r, 
                         sigma, T, N, proportional_cost, cl, batch_size=batch_size, 
                         epochs=1, device=device)

In [ ]:
# Compare strategies
agent_pnl, mjd_pnl, agent_sv, bs_sv = compare_strategies(agent, evalpaths, evaloption_prices, 
                                                        evaloption_deltas, K, r, 
                                                        sigma, T, N, proportional_cost, 
                                                        batch_size=batch_size, device=device)

In [ ]:
# Evaluate performance
print(f"Agent PnL: {np.mean(agent_pnl[:, -1])}")
print(f"MJD PnL: {np.mean(mjd_pnl[:, -1])}")

In [ ]:
# Example usage:
plot_pnl_histogram(agent_pnl[:, -1], mjd_pnl[:, -1], 'deep_hedging_mjd_lstm.png', False)

In [ ]:
def train_agent_full(train_cost_pct, calculate_loss, paths, option_prices, option_deltas, K, r, 
            sigma, T, N, batch_size=batch_size, device=device):
    agent = HedgingAgent().to(device)
    optimizer = optim.Adam(agent.parameters(), lr=0.01)
    p_cost = lambda x: proportional_cost(x, train_cost_pct)
    train_loss = train_agent(agent, optimizer, paths, option_prices, option_deltas, K, r, 
            sigma, T, N, p_cost, calculate_loss=calculate_loss, batch_size=batch_size, device=device)
    return agent

In [ ]:
def evaluate_agent(agent, evalpaths, evaloption_prices, evaloption_deltas, K, r, 
                   sigma, T, N, cost_pct, batch_size=batch_size, device=device):
    p_cost = lambda x: proportional_cost(x, cost_pct)
    agent_pnl, bs_pnl, _, _ = compare_strategies(agent, evalpaths, evaloption_prices, evaloption_deltas, K, r, 
                                       sigma, T, N, p_cost, batch_size=batch_size, device=device)
    return agent_pnl, bs_pnl

In [ ]:
def traineval_allagents(filename, params, S0, mu, sigma, T, M, N, batch_size=batch_size, device=device):
    columns = ['loss_type', 'alpha', 'lmda', 'cost_pct', 'agent_pl', 'mjd_pl']
    rows = []  # <-- collect rows here
    
    # Generate asset paths using Monte Carlo simulation
    paths = generate_asset_paths_merton(S0, mu, sigma, T, M, N, lamb, mu_j, sigma_j)

    option_prices, option_deltas = get_option_prices_and_deltas(
        paths, K, r, sigma, T, N, mu_j, sigma_j, lamb, device=device
    )

    evalpaths = generate_asset_paths_merton(
        S0, mu, sigma, T, M, N, lamb, mu_j, sigma_j, seed=1001
    )

    evaloption_prices, evaloption_deltas = get_option_prices_and_deltas(
        evalpaths, K, r, sigma, T, N, mu_j, sigma_j, lamb, device=device
    )
    
    for iparams in params:
        iloss_type, ialpha, ilmda, cost_pct = iparams
        
        filename_full = (
            filename + '_loss_' + iloss_type + '_alpha_' + str(ialpha) +
            '_lmda_' + str(ilmda) + '_cost_pct' + str(cost_pct)
        )

        cl = lambda x: calculate_loss(x, iloss_type, ialpha, ilmda)

        agent = train_agent_full(
            cost_pct, cl, paths, option_prices, option_deltas,
            K, r, sigma, T, N, batch_size=batch_size, device=device
        )

        torch.save(agent.state_dict(), filename_full + '.pth')

        agent_pnl, bs_pnl = evaluate_agent(
            agent, evalpaths, evaloption_prices, evaloption_deltas,
            K, r, sigma, T, N, cost_pct,
            batch_size=batch_size, device=device
        )
        
        plot_pnl_histogram(agent_pnl[:, -1], bs_pnl[:, -1], filename_full + '.png', False)
        plot_pnl_histogram(agent_pnl[:, -1], bs_pnl[:, -1], filename_full + '_bw.png', True)
        
        # store row as dict
        rows.append({
            'loss_type': iloss_type,
            'alpha': ialpha,
            'lmda': ilmda,
            'cost_pct': cost_pct,
            'agent_pl': np.mean(agent_pnl[:, -1]),
            'mjd_pl': np.mean(bs_pnl[:, -1]),
        })
    
    df = pd.DataFrame(rows, columns=columns)
    return df

In [ ]:
params = [('mse', 0.95, 1.0, 0.0),
          ('mse', 0.95, 1.0, 0.01),
          ('mse', 0.95, 1.0, 0.05),
          ('mean-variance', 0.95, 1.0, 0.0),
          ('mean-variance', 0.95, 1.0, 0.01),
          ('mean-variance', 0.95, 1.0, 0.05),
          ('cvar', 0.95, 1.0, 0.0),
          ('cvar', 0.95, 1.0, 0.01),
          ('cvar', 0.95, 1.0, 0.05),
          ('cvar', 0.99, 1.0, 0.0),
          ('cvar', 0.99, 1.0, 0.01),
          ('cvar', 0.99, 1.0, 0.05),
          ('entropic', 0.95, 1.0, 0.0),
          ('entropic', 0.95, 1.0, 0.01),
          ('entropic', 0.95, 1.0, 0.05),
          ('entropic', 0.95, 10.0, 0.0),
          ('entropic', 0.95, 10.0, 0.01),
          ('entropic', 0.95, 10.0, 0.05)]

In [ ]:
filename = 'MJD_DeepHedging_LSTM'

In [ ]:
result_df = traineval_allagents(filename, params, S0, mu, sigma, T, M, N, batch_size=batch_size, device=device)

In [ ]:
result_df

In [ ]:
result_df.to_latex()